In [6]:
import os
import math
import json
import warnings
import itertools
from datetime import datetime
from typing import List, Tuple

warnings.filterwarnings("ignore")

In [7]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [8]:
import numpy as np 
import pandas as pd
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from scipy.optimize import minimize

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.utils.args import DefaultArgumentParser
from fintorch.utils.resample import resample_candles_df

In [9]:
string_args = [
    "--start-date", "2024-01-01",
    "--stop-date", "2024-06-01",
    "--time-frame", "60",
    "--symbol", "BTC-USDT"
]

args = DefaultArgumentParser.parse(args=string_args)

In [21]:
sdf = ONLINE_EXCHANGE.future.data.get_top_long_short_ratios_account_dataframe(symbol=args.symbol, time_frame=TimeFrame.MIN15)
sdf.insert(0, "datetime", [datetime.fromtimestamp(ts) for ts in sdf.index])

sdf

,datetime,ratio,long,short
timestamp,,,,
1712204100,2024-04-04 07:45:00,2.2031,0.6878,0.3122
1712205000,2024-04-04 08:00:00,2.2031,0.6878,0.3122
1712205900,2024-04-04 08:15:00,2.2134,0.6888,0.3112
1712206800,2024-04-04 08:30:00,2.2237,0.6898,0.3102
1712207700,2024-04-04 08:45:00,2.2310,0.6905,0.3095
...,...,...,...,...
1712649600,2024-04-09 11:30:00,1.2080,0.5471,0.4529
1712650500,2024-04-09 11:45:00,1.2779,0.5610,0.4390
1712651400,2024-04-09 12:00:00,1.3137,0.5678,0.4322


In [34]:
df = sdf.copy()
destination_timeframe = TimeFrame.HOUR1
source_timeframe = TimeFrame.MIN15
window_size = destination_timeframe // source_timeframe
first_timestamp = math.ceil(df.index[0] / destination_timeframe) * destination_timeframe

print(window_size)
print(first_timestamp)


df = df[first_timestamp <= df.index]
indexer = pd.api.indexers.FixedForwardWindowIndexer(window_size=window_size)

df['first'] = 0 == (df.index % destination_timeframe)
df['new-ratio'] = df.ratio.rolling(window=indexer, min_periods=1, step=window_size).agg(lambda items: list(items)[0])
df['new-long'] = df.long.rolling(window=indexer, min_periods=1, step=window_size).agg(lambda items: list(items)[0])
df['new-short'] = df.short.rolling(window=indexer, min_periods=1, step=window_size).agg(lambda items: list(items)[0])
df = df.dropna()

df[["datetime", "first", "ratio", "new-ratio", "long", "new-long", "short", "new-short"]]

4
1712206800


,datetime,first,ratio,new-ratio,long,new-long,short,new-short
timestamp,,,,,,,,
1712206800,2024-04-04 08:30:00,True,2.2237,2.2237,0.6898,0.6898,0.3102,0.3102
1712210400,2024-04-04 09:30:00,True,2.2520,2.2520,0.6925,0.6925,0.3075,0.3075
1712214000,2024-04-04 10:30:00,True,2.2723,2.2723,0.6944,0.6944,0.3056,0.3056
1712217600,2024-04-04 11:30:00,True,2.1387,2.1387,0.6814,0.6814,0.3186,0.3186
1712221200,2024-04-04 12:30:00,True,2.0979,2.0979,0.6772,0.6772,0.3228,0.3228
...,...,...,...,...,...,...,...,...
1712638800,2024-04-09 08:30:00,True,1.1115,1.1115,0.5264,0.5264,0.4736,0.4736
1712642400,2024-04-09 09:30:00,True,1.1327,1.1327,0.5311,0.5311,0.4689,0.4689
1712646000,2024-04-09 10:30:00,True,1.1459,1.1459,0.5340,0.5340,0.4660,0.4660
